In [1]:
import pandas as pd 
import numpy as np
import glob
from sklearn.utils import resample
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

#Load CSV files & Combine into one DataFrame
files = glob.glob('../data/raw/*.csv')
df = pd.concat([pd.read_csv(f) for f in files], ignore_index = True)

#Remove leading/trailing whitespace from column names
df.columns = df.columns.str.strip() 

print(df.shape)
print((df['Label'].value_counts()))

(2830743, 79)
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [2]:
#Replace inf values with NaN
df.replace([np.inf, -np.inf], np.nan, inplace = True)

#Check for missing values
print(df.isnull().sum()[df.isnull().sum() > 0])

Flow Bytes/s      2867
Flow Packets/s    2867
dtype: int64


In [3]:
#Check percentage of missing values
null_pct = df.isnull().sum() / len(df) * 100
print(null_pct[null_pct > 0])

#Drop rows with missing values
df.dropna(inplace = True)
print(f"Shape after dropping nulls: {df.shape}")


Flow Bytes/s      0.101281
Flow Packets/s    0.101281
dtype: float64
Shape after dropping nulls: (2827876, 79)


In [4]:
#Check for duplicates and remove them
print(f"Duplicates: {df.duplicated().sum()}")
df.drop_duplicates(inplace = True)

print(f"Shape after dropping dupes: {df.shape}")

Duplicates: 307078
Shape after dropping dupes: (2520798, 79)


In [5]:
# Remove leading/trailing whitespace from 'Label' column
df['Label'] = df['Label'].str.strip() 

print(df['Label'].value_counts())

Label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [6]:
#Fix broken labels in 'Label' column
df['Label'] = df['Label'].str.replace(r'Web Attack.*Brute Force', 'Web Attack - Brute Force', regex = True)
df['Label'] = df['Label'].str.replace(r'Web Attack.*XSS', 'Web Attack - XSS', regex = True)
df['Label'] = df['Label'].str.replace(r'Web Attack.*Sql Injection', 'Web Attack - SQL Injection', regex = True)

print("After label fixes:")
print(df['Label'].value_counts())

After label fixes:
Label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack - Brute Force         1470
Web Attack - XSS                  652
Infiltration                       36
Web Attack - SQL Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [7]:
#Seperatre features and labels for SMOTE
x = df.drop(columns = 'Label')
y = df['Label']

#Encode labeels as integers for SMOTE
le = LabelEncoder()
y_encoded = le.fit_transform(y)

#Apply SMOTE to balance classes
sampling_strategy = {}
min_sample_size = 1000

unique_labels, counts = np.unique(y_encoded, return_counts = True)

for label, count in zip(unique_labels, counts):
    if count < min_sample_size:
        sampling_strategy[label] = min_sample_size

print(f"Classes to oversample: {sampling_strategy}")
print(f"Mapped Labels: {dict(zip(le.classes_, le.transform(le.classes_)))}")

smote = SMOTE(sampling_strategy = sampling_strategy, random_state = 42)
x_resampled, y_resampled = smote.fit_resample(x, y_encoded)

df = pd.DataFrame(x_resampled, columns = x.columns)
df['Label'] = le.inverse_transform(y_resampled)

print(f"\nAfter Smote:")
print(df['Label'].value_counts())


Classes to oversample: {np.int64(8): 1000, np.int64(9): 1000, np.int64(13): 1000, np.int64(14): 1000}
Mapped Labels: {'BENIGN': np.int64(0), 'Bot': np.int64(1), 'DDoS': np.int64(2), 'DoS GoldenEye': np.int64(3), 'DoS Hulk': np.int64(4), 'DoS Slowhttptest': np.int64(5), 'DoS slowloris': np.int64(6), 'FTP-Patator': np.int64(7), 'Heartbleed': np.int64(8), 'Infiltration': np.int64(9), 'PortScan': np.int64(10), 'SSH-Patator': np.int64(11), 'Web Attack - Brute Force': np.int64(12), 'Web Attack - SQL Injection': np.int64(13), 'Web Attack - XSS': np.int64(14)}

After Smote:
Label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack - Brute Force         1470
Inf

In [8]:
#Group low occuring attack types into 'Other Attack'
df['Label'] = df['Label'].replace({
    'Heartbleed': 'Other Attack',
    'Infiltration': 'Other Attack',
    'Web Attack - SQL Injection': 'Other Attack',
})

print("After grouping low occuring attacks:")
print(df['Label'].value_counts())

After grouping low occuring attacks:
Label
BENIGN                      2095057
DoS Hulk                     172846
DDoS                         128014
PortScan                      90694
DoS GoldenEye                 10286
FTP-Patator                    5931
DoS slowloris                  5385
DoS Slowhttptest               5228
SSH-Patator                    3219
Other Attack                   3000
Bot                            1948
Web Attack - Brute Force       1470
Web Attack - XSS               1000
Name: count, dtype: int64


In [9]:
# Drop columns with only one unique value
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
print(f"Dropping constant columns: {constant_cols}")

df.drop(columns = constant_cols, inplace = True)

print(f"Shape after dropping constant columns: {df.shape}")

Dropping constant columns: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Shape after dropping constant columns: (2524078, 71)


In [10]:
#Dataset Balancing 
print("Before balance:")
print(df['Label'].value_counts())

benign = df[df['Label'] == 'BENIGN']
attacks = df[df['Label'] != 'BENIGN']

benign_downsampled = resample(
    benign,
    n_samples = 500000,
    random_state = 42,
    replace = False
)

df = pd.concat([benign_downsampled, attacks])

print("\nAfter balance:")
print(df['Label'].value_counts())
print(f"\nClass proportions:")
print(df['Label'].value_counts(normalize = True) * 100)

Before balance:
Label
BENIGN                      2095057
DoS Hulk                     172846
DDoS                         128014
PortScan                      90694
DoS GoldenEye                 10286
FTP-Patator                    5931
DoS slowloris                  5385
DoS Slowhttptest               5228
SSH-Patator                    3219
Other Attack                   3000
Bot                            1948
Web Attack - Brute Force       1470
Web Attack - XSS               1000
Name: count, dtype: int64

After balance:
Label
BENIGN                      500000
DoS Hulk                    172846
DDoS                        128014
PortScan                     90694
DoS GoldenEye                10286
FTP-Patator                   5931
DoS slowloris                 5385
DoS Slowhttptest              5228
SSH-Patator                   3219
Other Attack                  3000
Bot                           1948
Web Attack - Brute Force      1470
Web Attack - XSS              1000
Name: c

In [11]:
df = df.sample(frac = 1, random_state = 42).reset_index(drop = True)

print(f"Final shape: {df.shape}")
print(f"Current Nulls: {df.isnull().sum().sum()}")
print(f"Label distribution:\n {df['Label'].value_counts()}")

expected_labels = [
    'BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk',
    'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator',
    'Other Attack', 'PortScan', 'SSH-Patator',
    'Web Attack - Brute Force', 'Web Attack - XSS'
]

missing = [l for l in expected_labels if l not in df['Label'].values]
if missing:
    print(f"\n[WARNING] Missing labels: {missing}")
else:
    print("\nAll expected labels present")


Final shape: (929021, 71)
Current Nulls: 0
Label distribution:
 Label
BENIGN                      500000
DoS Hulk                    172846
DDoS                        128014
PortScan                     90694
DoS GoldenEye                10286
FTP-Patator                   5931
DoS slowloris                 5385
DoS Slowhttptest              5228
SSH-Patator                   3219
Other Attack                  3000
Bot                           1948
Web Attack - Brute Force      1470
Web Attack - XSS              1000
Name: count, dtype: int64

All expected labels present


In [12]:
df.to_csv('../data/CIC-IDS-17_CLEANED.csv', index = False)
print("Saved cleaned dataset")

Saved cleaned dataset
